# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/barak-hub/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:

import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

token = userdata.get("HF_TOKEN")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_query_90d.parquet",
    repo_type="dataset",
    token=token
)

df = pd.read_parquet(file_path)

print("Dataset loaded successfully!")
print("Total rows:", len(df))
print("Unique clients:", df["client_hash_id"].nunique())
print(
    "Unique webpages:",
    df[["client_hash_id", "content_hash_id"]]
    .drop_duplicates()
    .shape[0]
)


fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Dataset loaded successfully!
Total rows: 2414248
Unique clients: 52
Unique webpages: 133852


In [2]:

# Week 7 — Aggregate query data to webpage level

page_df = (
    df.groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        impressions_prev30=("impressions_prev30", "sum"),
        impressions_last30=("impressions_last30", "sum"),
        clicks_prev30=("clicks_prev30", "sum"),
        query_count=("query_hash_id", "nunique")
    )
)

# Calculate observed impression decline
page_df["declining_impressions"] = (
    page_df["impressions_last30"]
    < page_df["impressions_prev30"]
).astype(int)

# Calculate absolute and percentage change
page_df["impression_change"] = (
    page_df["impressions_last30"]
    - page_df["impressions_prev30"]
)

page_df["decline_pct"] = (
    100
    * (page_df["impressions_prev30"] - page_df["impressions_last30"])
    / page_df["impressions_prev30"].where(
        page_df["impressions_prev30"] > 0
    )
)

print("Total webpages:", len(page_df))
print("Declining webpages:",
      page_df["declining_impressions"].sum())
print("Non-declining webpages:",
      (page_df["declining_impressions"] == 0).sum())

print("\nPreview:")
print(
    page_df[
        ["impressions_prev30",
         "impressions_last30",
         "impression_change",
         "decline_pct"]
    ].head()
)


Total webpages: 133852
Declining webpages: 80534
Non-declining webpages: 53318

Preview:
   impressions_prev30  impressions_last30  impression_change  decline_pct
0                   0                  16                 16          NaN
1                 139                 123                -16    11.510791
2                  59                  38                -21    35.593220
3                   0                  28                 28          NaN
4                   9                   6                 -3    33.333333


In [3]:

# Week 7 — Section 1: Ranked SEO Action Queue

# Only webpages with observed declines
declining_pages = page_df[
    page_df["declining_impressions"] == 1
].copy()

# Absolute number of impressions lost
declining_pages["impressions_lost"] = (
    -declining_pages["impression_change"]
)

# Create transparent reason codes
def assign_reason_codes(row):
    reasons = []

    if row["impressions_lost"] >= 100:
        reasons.append("HIGH_VOLUME_LOSS")

    if row["decline_pct"] >= 30:
        reasons.append("LARGE_PERCENT_DROP")

    if row["impressions_prev30"] < 20:
        reasons.append("LOW_BASELINE")

    if not reasons:
        reasons.append("MODERATE_DECLINE")

    return ", ".join(reasons)

declining_pages["reason_codes"] = (
    declining_pages.apply(assign_reason_codes, axis=1)
)

# Rank by absolute impression loss
action_queue = (
    declining_pages.sort_values(
        "impressions_lost",
        ascending=False
    )
    .reset_index(drop=True)
)

action_queue["priority_rank"] = (
    action_queue.index + 1
)

# Display top 10 without client identifiers
display(
    action_queue[
        [
            "priority_rank",
            "impressions_prev30",
            "impressions_last30",
            "impressions_lost",
            "decline_pct",
            "reason_codes"
        ]
    ].head(10)
)

print("Total webpages requiring review:", len(action_queue))


,priority_rank,impressions_prev30,impressions_last30,impressions_lost,decline_pct,reason_codes
0,1,177963,6622,171341,96.279002,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP"
1,2,185602,20756,164846,88.816931,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP"
2,3,118496,595,117901,99.497873,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP"
3,4,131836,25005,106831,81.033253,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP"
4,5,133285,41887,91398,68.573358,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP"
5,6,108461,27203,81258,74.919095,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP"
6,7,80721,11709,69012,85.494481,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP"
7,8,74165,9100,65065,87.730061,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP"
8,9,110973,46809,64164,57.819470,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP"
9,10,79458,18172,61286,77.130056,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP"


Total webpages requiring review: 80534


In [4]:

# Week 7 — Map reason codes to recommended review actions

def recommend_action(reason_codes):
    actions = []

    if "HIGH_VOLUME_LOSS" in reason_codes:
        actions.append(
            "Investigate major search visibility loss"
        )

    if "LARGE_PERCENT_DROP" in reason_codes:
        actions.append(
            "Review search queries, ranking changes, "
            "and indexing status"
        )

    if "LOW_BASELINE" in reason_codes:
        actions.append(
            "Verify data volume before drawing conclusions"
        )

    if "MODERATE_DECLINE" in reason_codes:
        actions.append(
            "Monitor performance and investigate if decline persists"
        )

    return "; ".join(actions)

action_queue["recommended_action"] = (
    action_queue["reason_codes"].apply(recommend_action)
)

display(
    action_queue[
        [
            "priority_rank",
            "impressions_lost",
            "decline_pct",
            "reason_codes",
            "recommended_action"
        ]
    ].head(10)
)

print(
    "Actions assigned:",
    action_queue["recommended_action"].notna().sum()
)


,priority_rank,impressions_lost,decline_pct,reason_codes,recommended_action
0,1,171341,96.279002,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP",Investigate major search visibility loss; Revi...
1,2,164846,88.816931,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP",Investigate major search visibility loss; Revi...
2,3,117901,99.497873,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP",Investigate major search visibility loss; Revi...
3,4,106831,81.033253,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP",Investigate major search visibility loss; Revi...
4,5,91398,68.573358,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP",Investigate major search visibility loss; Revi...
5,6,81258,74.919095,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP",Investigate major search visibility loss; Revi...
6,7,69012,85.494481,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP",Investigate major search visibility loss; Revi...
7,8,65065,87.730061,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP",Investigate major search visibility loss; Revi...
8,9,64164,57.819470,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP",Investigate major search visibility loss; Revi...
9,10,61286,77.130056,"HIGH_VOLUME_LOSS, LARGE_PERCENT_DROP",Investigate major search visibility loss; Revi...


Actions assigned: 80534


### Section 1 — Ranked Actions and Reason Codes

**Objective**

I developed an explainable SEO action queue to prioritize webpages that experienced declines in Google Search impressions.

**Methodology**

I aggregated the query-level dataset into 133,852 unique client-webpage records. Of these, 80,534 webpages experienced a decline in impressions between the previous and last 30-day reporting periods.

I ranked declining webpages by the absolute number of impressions lost, prioritizing the largest observed losses.

**Reason Codes**

| Reason Code | Definition |
|---|---|
| `HIGH_VOLUME_LOSS` | At least 100 impressions lost |
| `LARGE_PERCENT_DROP` | At least 30% decline |
| `LOW_BASELINE` | Fewer than 20 impressions in the previous period |
| `MODERATE_DECLINE` | Decline that does not meet the other thresholds |

These thresholds are initial operational rules rather than statistically validated decision boundaries. A webpage can receive multiple reason codes.

**Results**

The system assigned review recommendations to all 80,534 declining webpages.

The highest-ranked webpage lost 171,341 impressions, representing a 96.28% decline. Its reason codes were `HIGH_VOLUME_LOSS` and `LARGE_PERCENT_DROP`.

**Recommended Actions**

High-volume losses trigger an investigation of search visibility. Large percentage declines trigger reviews of search queries, rankings, and indexing status. Low-baseline cases require additional data verification, while moderate declines are candidates for continued monitoring.

**Limitations**

The queue is based on observed historical changes and transparent ranking rules. It does not establish the causes of declines or predict future losses. Recommendations require human review before any SEO changes are implemented.

**Conclusion**

The action queue converts measured search-performance changes into prioritized, explainable investigation tasks. It is intended to support SEO analysts rather than automate decisions.

### Section 2 — Intended Use and Limits

**Intended Users**

The SEO action queue is designed for SEO analysts, content strategists, and digital marketing teams responsible for monitoring website search performance.

**Intended Use**

The system supports the following activities:

1. Identifying webpages with observed declines in Google Search impressions.
2. Prioritizing webpages for investigation based on absolute impression losses.
3. Explaining review priorities through transparent reason codes.
4. Suggesting investigation steps, such as reviewing search queries, rankings, and indexing status.
5. Supporting human decisions about which webpages require attention.

**System Limitations**

The system uses historical search-performance measurements from a single reporting window. It does not establish why impressions declined or guarantee that similar declines will occur in the future.

The priority ranking is rule-based, using observed impression losses rather than model-generated risk probabilities.

The thresholds used for reason codes are initial operational choices and have not been independently validated.

The Week 5 Decision Tree achieved an F1-score of 0.8185 under client-grouped validation, but that result does not establish future forecasting performance.

**Prohibited or Unsupported Uses**

The system should not be used to:

- Automatically delete, rewrite, or publish webpage content.
- Claim that a particular SEO issue caused a decline without investigation.
- Guarantee improvements in search rankings or traffic.
- Make high-impact business decisions without human review.
- Present observed historical declines as certain future predictions.

**Human Oversight**

An SEO analyst should review the evidence, verify the context, and approve any proposed changes before implementation.

**Conclusion**

The action queue is a decision-support tool that helps teams prioritize SEO investigations. It is not an autonomous SEO optimization system or a validated future-performance forecasting tool.

### Section 3 — Human Review and the No-Go List

**Objective**

Establish a human-review process that ensures SEO recommendations are verified, explainable, and approved before implementation.

**Human Review Workflow**

1. **Identify:** Select a high-priority webpage from the ranked action queue.
2. **Verify:** Confirm the observed impression decline using Google Search Console or another reliable reporting source.
3. **Investigate:** Review affected search queries, indexing status, ranking changes, content updates, and possible technical issues.
4. **Assess:** Determine whether the evidence supports a specific corrective action or whether further investigation is required.
5. **Approve:** Obtain authorization from the responsible SEO analyst or content manager before implementing changes.
6. **Monitor:** Record the action taken and compare subsequent search-performance measurements against the pre-action baseline.

**No-Go List — Actions the System Must Not Take Automatically**

| Prohibited Action | Reason |
|---|---|
| Delete or unpublish webpages | Could remove valuable content or business assets |
| Rewrite content automatically | May introduce factual errors or damage search relevance |
| Change canonical tags or redirects | Could cause indexing and traffic problems |
| Modify robots.txt or noindex settings | Could unintentionally block search visibility |
| Guarantee ranking improvements | The available evidence does not support guaranteed outcomes |
| Attribute declines to a specific cause without verification | Observed correlation does not establish causation |
| Expose client identifiers or confidential data | Violates data confidentiality expectations |

**Escalation Rules**

High-impact changes, uncertain diagnoses, conflicting evidence, and potentially sensitive client information must be escalated to an authorized reviewer.

**Audit Trail**

For each reviewed webpage, the team should record its priority rank, reason codes, supporting evidence, proposed action, reviewer decision, approval status, and follow-up results.

**Conclusion**

The action queue supports investigation and prioritization, but human reviewers retain responsibility for diagnosing problems and approving SEO changes. This approach reduces the risk of harmful automation and unsupported conclusions.

### Section 4 — Monitoring and Retraining Triggers

**Objective**

Establish a monitoring framework to detect changes in data quality, model performance, and SEO action-queue reliability.

**Monitoring Framework**

| Monitoring Area | Metric | Proposed Trigger | Response |
|---|---|---|---|
| Data quality | Missing feature values | Any unexpected missing values | Investigate data ingestion |
| Data drift | Change in feature distributions | Significant deviation from training baseline | Investigate distribution changes |
| Model performance | F1-score | Drop greater than 5 percentage points from baseline | Review model performance and consider retraining |
| Prediction quality | Precision and recall | Sustained deterioration | Investigate false positives and false negatives |
| Client coverage | New or unseen clients | Substantial increase in unseen-client volume | Review generalization |
| Action queue | Reason-code distribution | Unexpected changes in category frequencies | Audit thresholds and source data |
| Data freshness | Reporting-window completion | Missing or delayed reporting data | Delay or flag affected evaluations |

These triggers are proposed operational guidelines and require validation before production deployment.

**Baseline Performance**

The Week 5 Decision Tree Classifier achieved the following results under client-grouped validation:

- Precision: 0.7262
- Recall: 0.9378
- F1-score: 0.8185

These measurements provide an initial reference for future evaluations. Comparisons should use consistent target definitions and comparable validation procedures.

**Retraining Procedure**

1. Collect and validate newly available reporting data.
2. Confirm that features are available before the outcome period.
3. Check for missing values, schema changes, and data drift.
4. Evaluate the existing model on a new, chronologically separated test period.
5. Compare performance against the established baseline.
6. Retrain if evidence indicates that the model no longer meets operational requirements.
7. Validate the candidate model before deployment and obtain human approval.

**Important Limitations**

The current experiment uses a single historical reporting window. It has not established a reliable future-performance baseline.

A monitoring alert should trigger investigation rather than automatic retraining. Retraining is appropriate only when sufficient new data and valid evaluation evidence are available.

The rule-based action queue must also be monitored independently because changes in reason-code frequencies do not necessarily indicate model drift.

**Conclusion**

Monitoring helps maintain the reliability of the ML model and the SEO action queue. Retraining decisions should be evidence-based, validated on new data, and subject to human oversight.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [5]:

# Week 7 — Section 5: Export Action Queue

import os
import pandas as pd

# Create export directory
os.makedirs("exports", exist_ok=True)

# Export only analysis fields, excluding client identifiers
export_columns = [
    "priority_rank",
    "impressions_prev30",
    "impressions_last30",
    "impressions_lost",
    "decline_pct",
    "reason_codes",
    "recommended_action"
]

export_df = action_queue[export_columns].copy()

# Save CSV
export_path = "exports/w07_seo_action_queue.csv"
export_df.to_csv(export_path, index=False)

print("Export successful!")
print("File:", export_path)
print("Rows exported:", len(export_df))
print("Columns exported:", len(export_df.columns))
print("File exists:", os.path.exists(export_path))


Export successful!
File: exports/w07_seo_action_queue.csv
Rows exported: 80534
Columns exported: 7
File exists: True


In [6]:

# Week 7 — Research summary export

summary = pd.DataFrame([{
    "total_webpages": len(page_df),
    "declining_webpages": len(action_queue),
    "decline_rate_pct": round(
        100 * len(action_queue) / len(page_df), 2
    ),
    "highest_observed_loss": int(
        action_queue["impressions_lost"].max()
    ),
    "week5_grouped_precision": 0.7262,
    "week5_grouped_recall": 0.9378,
    "week5_grouped_f1": 0.8185,
    "validation_method": "Client-grouped holdout",
    "interpretation": (
        "Historical observed-decline analysis; "
        "not validated for future forecasting"
    )
}])

summary_path = "exports/w07_research_summary.csv"
summary.to_csv(summary_path, index=False)

print("Research summary exported successfully!")
print("File:", summary_path)
print("Rows:", len(summary))
print("Columns:", len(summary.columns))

display(summary.T)


Research summary exported successfully!
File: exports/w07_research_summary.csv
Rows: 1
Columns: 9


,0
total_webpages,133852
declining_webpages,80534
decline_rate_pct,60.17
highest_observed_loss,171341
week5_grouped_precision,0.7262
week5_grouped_recall,0.9378
week5_grouped_f1,0.8185
validation_method,Client-grouped holdout
interpretation,Historical observed-decline analysis; not vali...


In [7]:

# Download Week 7 research deliverables

from google.colab import files

files.download("exports/w07_seo_action_queue.csv")
files.download("exports/w07_research_summary.csv")

print("Both Week 7 exports are ready for download.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Both Week 7 exports are ready for download.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.